<a href="https://colab.research.google.com/github/ekonjmrivas-devops/llm_engineering/blob/mis-ejercicios/W3_PRACTICA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Semana 3 - Práctica


**Bloque - Librerías**

In [2]:
import os
import requests
from IPython.display import Markdown, display, update_display
from openai import OpenAI
from google.colab import drive
from huggingface_hub import login
from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
import torch
from datetime import datetime

Conexión con Google Drive
- Montar Drive en Colab
- Listar/Seleccionar archivo de origen
- Guardar archivo de salida

**Bloque - Setup y conexión a Drive**

In [3]:
# Conexión a Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#drive.flush_and_unmount()
#drive.mount('/content/drive', force_remount=True)

In [4]:
# Rutas fijas del proyecto
BASE_PATH = '/content/drive/MyDrive/cursollms/week3'
INPUT_PATH = f'{BASE_PATH}/Inbound files'
OUTPUT_PATH = f'{BASE_PATH}/Outbound files'

In [5]:
# Crear carpetas si no existen
os.makedirs(INPUT_PATH, exist_ok=True)
os.makedirs(OUTPUT_PATH, exist_ok=True)

In [6]:
print(f"Carpeta de entrada: {INPUT_PATH}")
print(f"Carpeta de salida: {OUTPUT_PATH}")

Carpeta de entrada: /content/drive/MyDrive/cursollms/week3/Inbound files
Carpeta de salida: /content/drive/MyDrive/cursollms/week3/Outbound files


**Bloque - Listar archivos de la carpeta de entrada**

In [7]:
# Listar archivos disponibles en directorio de Drive

def listar_archivos_origen():
    """
    Lista los archivos disponibles en la carpeta de entrada de Drive.
    Devuelve una lista de nombres de archivo (sin la ruta completa),
    pensada para poblar el Dropdown de Gradio.
    """
    extensiones_validas = ('.mp3', '.wav', '.m4a', '.txt')

    archivos = [
        f for f in os.listdir(INPUT_PATH)
        if f.lower().endswith(extensiones_validas)
    ]

    archivos.sort()
    return archivos

In [8]:
# Comprobación de la función
archivos_disponibles = listar_archivos_origen()
print(f"Archivos encontrados: {len(archivos_disponibles)}")
for a in archivos_disponibles:
    print(f"  - {a}")

Archivos encontrados: 2
  - Desarrollo de un caso práctico.txt
  - denver_extract.mp3


In [9]:
# Diagnóstico: ver TODO lo que hay en la carpeta, sin filtrar por extensión
print(os.listdir(INPUT_PATH))

['Irregular Verbs.pdf', 'training-5-azure-devops-pipelines.md', 'Desarrollo de un caso práctico.txt', 'denver_extract.mp3']


Entrada y detección de tipo
- Audio > Whisper (Transcripción)
- Texto > Lectura directa

**Bloque - Transcripción de audio con Whisper (OpenAI API)**

In [10]:
# Crear cliente de OpenAI
client = OpenAI(api_key= userdata.get('OPENAI_API_KEY'))
AUDIO_MODEL = "whisper-1"

In [11]:
def transcribir_audio(nombre_archivo):
    """
    Transcribe un archivo de audio a texto usando Whisper (OpenAI API).
    nombre_archivo: nombre del archivo dentro de INPUT_PATH (ej. 'denver_extract.mp3')
    Devuelve: el texto transcrito.
    """
    ruta_completa = os.path.join(INPUT_PATH, nombre_archivo)

    with open(ruta_completa, "rb") as audio_file:
        transcripcion = client.audio.transcriptions.create(
            model=AUDIO_MODEL,
            file=audio_file,
            response_format="text"
        )

    return transcripcion

In [12]:
texto_transcrito = transcribir_audio("denver_extract.mp3")
print(texto_transcrito[:500])
print(f"\nLongitud total: {len(texto_transcrito)} caracteres")

and kind of the confluence of this whole idea of the confluence week, the merging of two rivers and as we've kind of seen recently in politics and in the world, there's a lot of situations where water is very important right now and it's a very big issue. So that is the reason that the back of the logo is considered water. So let me see the creation of the logo here. So that basically kind of sums up the reason behind the logo and all the meanings behind the symbolism and you'll hear a little bi

Longitud total: 12521 caracteres


**Bloque - Lectura de archivo de texto**

In [13]:
def leer_texto(nombre_archivo):
    """
    Lee directamente un archivo de texto (.txt) de la carpeta de entrada.
    nombre_archivo: nombre del archivo dentro de INPUT_PATH (ej. 'Desarrollo de un caso práctico.txt')
    Devuelve: el contenido del archivo como string.
    """
    ruta_completa = os.path.join(INPUT_PATH, nombre_archivo)

    with open(ruta_completa, "r", encoding="utf-8") as f:
        contenido = f.read()

    return contenido

In [14]:
texto_leido = leer_texto("Desarrollo de un caso práctico.txt")
print(texto_leido[:500])
print(f"\nLongitud total: {len(texto_leido)} caracteres")

Apertura: domingo, 5 de abril de 2026, 00:00
Cierre: martes, 21 de abril de 2026, 00:00
Se pide desarrollar dos ejercicios (uno de terraform y otro de ansible) para comprobar si se han entendido de forma correcta los conceptos explicados en el módulo

Ejercicios_Modulo4_IaC.pdf Ejercicios_Modulo4_IaC.pdf21 de marzo de 2026, 13:38


Longitud total: 332 caracteres


Procesamiento con LLM
- Resumen del acta/minuta
- Extracción de acciones/próximos pasos

Generación del archivo de salida
- Nombre automático
- Formato TXT o PDF
- Guardado en Drive

Interfaz Gradio
- Formulario de inputs
- Preview de resultado (scroll)
- Previo del archivo de salida
